### Binary classification of IMDB movie reviews

In [12]:
import os 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# Sharper plots
%config InlineBackend.figure_format = "retina"

from sklearn.datasets import load_files
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
import tarfile
from io import BytesIO
import requests

url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

def load_imdb_dataset(extract_path, overwrite=False):
    # check if it exists already
    if(
        os.path.isfile(os.path.join(extract_path, "aclImdb", "README"))
        and not overwrite
    ):
        print("IMDB dataset already in place.")
        return
    
    print("downloading dataset from: ", url)
    response = requests.get(url)

    tar = tarfile.open(mode="r:gz", fileobj=BytesIO(response.content))

    data = tar.extractall(extract_path)

In [3]:
DATA_PATH = "."
load_imdb_dataset(DATA_PATH)

IMDB dataset already in place.


In [4]:
PATH_TO_IMDB = "./aclImdb/train"
os.path.exists(PATH_TO_IMDB)
os.listdir("./aclImdb/test")

['labeledBow.feat', 'neg', 'pos', 'urls_neg.txt', 'urls_pos.txt']

In [14]:
""" from pathlib import Path
from tqdm.notebook import tqdm

PATH_TO_IMDB_TRAIN = "./aclImdb/train"
PATH_TO_IMDB_TEST = "./aclImdb/test"

texts = []
labels = []

for label, sentiment in enumerate(["neg", "pos"]):
    files = list(Path(PATH_TO_IMDB_TRAIN, sentiment).glob("*.txt"))

    for file in tqdm(files, desc=f"Loading {sentiment}"):
        texts.append(file.read_text(encoding="utf-8"))
        labels.append(label) """

' from pathlib import Path\nfrom tqdm.notebook import tqdm\n\nPATH_TO_IMDB_TRAIN = "./aclImdb/train"\nPATH_TO_IMDB_TEST = "./aclImdb/test"\n\ntexts = []\nlabels = []\n\nfor label, sentiment in enumerate(["neg", "pos"]):\n    files = list(Path(PATH_TO_IMDB_TRAIN, sentiment).glob("*.txt"))\n\n    for file in tqdm(files, desc=f"Loading {sentiment}"):\n        texts.append(file.read_text(encoding="utf-8"))\n        labels.append(label) '

In [4]:
# change if you have it in alternative location
# slow process, so may take some time to run
PATH_TO_IMDB_TRAIN = "./aclImdb/train"
PATH_TO_IMDB_TEST = "./aclImdb/test"

reviews_train = load_files(
    PATH_TO_IMDB_TRAIN, categories=["pos", "neg"]
)
text_train, y_train = reviews_train.data, reviews_train.target

reviews_test = load_files(PATH_TO_IMDB_TEST, categories=["pos", "neg"])
text_test, y_test = reviews_test.data, reviews_test.target

In [11]:
reviews_train["data"][:5] , reviews_train["target"][:5]

([b"Zero Day leads you to think, even re-think why two boys/young men would do what they did - commit mutual suicide via slaughtering their classmates. It captures what must be beyond a bizarre mode of being for two humans who have decided to withdraw from common civility in order to define their own/mutual world via coupled destruction.<br /><br />It is not a perfect movie but given what money/time the filmmaker and actors had - it is a remarkable product. In terms of explaining the motives and actions of the two young suicide/murderers it is better than 'Elephant' - in terms of being a film that gets under our 'rationalistic' skin it is a far, far better film than almost anything you are likely to see. <br /><br />Flawed but honest with a terrible honesty.",
  b'Words can\'t describe how bad this movie is. I can\'t explain it by writing only. You have too see it for yourself to get at grip of how horrible a movie really can be. Not that I recommend you to do that. There are so many c

In [13]:
df = pd.DataFrame({
    "review": reviews_train["data"][:5],
    "target": reviews_train["target"][:5]
})

print(df)

                                              review  target
0  b"Zero Day leads you to think, even re-think w...       1
1  b'Words can\'t describe how bad this movie is....       0
2  b'Everyone plays their part pretty well in thi...       1
3  b'There are a lot of highly talented filmmaker...       0
4  b'I\'ve just had the evidence that confirmed m...       0


In [9]:
print(reviews_train.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


In [ ]:
print(reviews_train.target_names)
# so 0 is negative and 1 is positive

['neg', 'pos']


In [6]:
print("Number of documents in training data: %d" % len(text_train))
print(np.bincount(y_train))
print("Number of documents in test data: %d" % len(text_test))
print(np.bincount(y_test))

Number of documents in training data: 25000
[12500 12500]
Number of documents in test data: 25000
[12500 12500]


In [7]:
print(text_train[1])

b'Words can\'t describe how bad this movie is. I can\'t explain it by writing only. You have too see it for yourself to get at grip of how horrible a movie really can be. Not that I recommend you to do that. There are so many clich\xc3\xa9s, mistakes (and all other negative things you can imagine) here that will just make you cry. To start with the technical first, there are a LOT of mistakes regarding the airplane. I won\'t list them here, but just mention the coloring of the plane. They didn\'t even manage to show an airliner in the colors of a fictional airline, but instead used a 747 painted in the original Boeing livery. Very bad. The plot is stupid and has been done many times before, only much, much better. There are so many ridiculous moments here that i lost count of it really early. Also, I was on the bad guys\' side all the time in the movie, because the good guys were so stupid. "Executive Decision" should without a doubt be you\'re choice over this one, even the "Turbulenc

In [ ]:
y_train[1]  # bad review

## 2. A simple word count

In [5]:
# create a dict of all the works using CountVectorizer

cv = CountVectorizer()
cv.fit(text_test)

len(cv.vocabulary_)

73822



## 1. Create some text data

```python
text_test = [
    "I love Python",
    "I love machine learning",
    "Python is awesome"
]
```

This is called a **corpus** (a collection of documents).

---

## 2. Create a CountVectorizer

```python
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
```

At this point, the vectorizer knows nothing about your text.

---

## 3. Fit the vectorizer

```python
cv.fit(text_test)
```

During `fit()`, CountVectorizer:

### Step A: Converts text to lowercase

```text
i love python
i love machine learning
python is awesome
```

### Step B: Tokenizes into words

```text
['i', 'love', 'python']
['i', 'love', 'machine', 'learning']
['python', 'is', 'awesome']
```

### Step C: Collects unique words

```text
i
love
python
machine
learning
is
awesome
```

These unique words become the **vocabulary**.

---

## 4. View the vocabulary

```python
print(cv.vocabulary_)
```

Output (indices may vary):

```python
{
    'love': 2,
    'python': 5,
    'machine': 3,
    'learning': 1,
    'is': 0,
    'awesome': 4
}
```

### What does this mean?

Each word gets an index.

| Word     | Index |
| -------- | ----- |
| is       | 0     |
| learning | 1     |
| love     | 2     |
| machine  | 3     |
| awesome  | 4     |
| python   | 5     |

Think of this as:

```python
word -> column number
```

---

## 5. Number of unique words

```python
len(cv.vocabulary_)
```

Output:

```python
6
```

Because there are 6 unique words.

---

## 6. Transform text into vectors

Now use:

```python
X = cv.transform(text_test)
```

Convert to array:

```python
print(X.toarray())
```

Output:

```python
[
 [0, 0, 1, 0, 0, 1],
 [0, 1, 1, 1, 0, 0],
 [1, 0, 0, 0, 1, 1]
]
```

---

### How is the first row created?

Sentence:

```text
"I love Python"
```

Vocabulary:

| Word     | Index |
| -------- | ----- |
| is       | 0     |
| learning | 1     |
| love     | 2     |
| machine  | 3     |
| awesome  | 4     |
| python   | 5     |

Count occurrences:

| Word     | Count |
| -------- | ----- |
| is       | 0     |
| learning | 0     |
| love     | 1     |
| machine  | 0     |
| awesome  | 0     |
| python   | 1     |

Vector:

```python
[0, 0, 1, 0, 0, 1]
```

---

### Second sentence

```text
"I love machine learning"
```

Counts:

| Word     | Count |
| -------- | ----- |
| is       | 0     |
| learning | 1     |
| love     | 1     |
| machine  | 1     |
| awesome  | 0     |
| python   | 0     |

Vector:

```python
[0, 1, 1, 1, 0, 0]
```

---

### Third sentence

```text
"Python is awesome"
```

Counts:

| Word     | Count |
| -------- | ----- |
| is       | 1     |
| learning | 0     |
| love     | 0     |
| machine  | 0     |
| awesome  | 1     |
| python   | 1     |

Vector:

```python
[1, 0, 0, 0, 1, 1]
```

---

## Visual summary

```python
text_test
│
├── "I love Python"
├── "I love machine learning"
└── "Python is awesome"
        │
        ▼
cv.fit(text_test)
        │
        ▼
Vocabulary
{
 'is':0,
 'learning':1,
 'love':2,
 'machine':3,
 'awesome':4,
 'python':5
}
        │
        ▼
cv.transform(text_test)
        │
        ▼
[
 [0,0,1,0,0,1],
 [0,1,1,1,0,0],
 [1,0,0,0,1,1]
]
```

This is why CountVectorizer is called **Bag of Words**: it ignores word order and simply counts how many times each vocabulary word appears in each document.


In [6]:
print(cv.get_feature_names_out()[:50])
print(cv.get_feature_names_out()[50000:50050])

['00' '000' '00000000000' '00000001' '000dm' '001' '0069' '007' '0079'
 '007s' '0083' '009' '00am' '00o' '00pm' '00s' '00schneider' '01' '0126'
 '0148' '02' '0230' '03' '039' '04' '044' '05' '05nomactr' '06' '0615'
 '07' '07b' '08' '08th' '09' '0and' '0f' '0tt' '10' '100' '1000' '10000'
 '1000000' '10000000000' '10000000000000' '10000th' '1000s' '1000th'
 '1001' '1004']
['polaroids' 'polars' 'pole' 'polec' 'poledouris' 'polemic' 'polemical'
 'polemics' 'poler' 'poles' 'polglase' 'poli' 'polic' 'police' 'policeman'
 'policemen' 'policewoman' 'policians' 'policier' 'policies' 'policing'
 'policy' 'polio' 'polish' 'polished' 'polishes' 'polishing' 'polite'
 'politely' 'politeness' 'politest' 'politic' 'political' 'politicall'
 'politically' 'politicans' 'politicial' 'politician' 'politicians'
 'politicize' 'politicka' 'politicking' 'politicly' 'politico' 'politics'
 'politions' 'polito' 'polivka' 'polizia' 'polizioteschi']
